In [1]:
!pip install accelerate demucs silero-vad bnunicodenormalizer jiwer noisereduce librosa peft

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.1/87.1 kB 6.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 5.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 103.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.1/249.1 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 5.4 MB/s eta 0:00:00
  Created wheel for demucs: filename=demucs-4.0.1-py3-none-any.whl size=78388 sha256=0130771bbcb331f2f7b8dcde64ee64af200fee1560f797c41242a23b41dde891
  

In [2]:
!pip install faster-whisper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 96.4 MB/s eta 0:00:00


In [3]:
# Generated from: whisper-e5-lora (1).ipynb
# Converted at: 2026-03-04T11:16:30.783Z
# Next step (optional): refactor into modules & generate tests with RunCell
# Quick start: pip install runcell

# !pip install accelerate demucs silero-vad bnunicodenormalizer jiwer noisereduce librosa peft


# ## 2. Import Libraries


import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import os
import re
import csv
import time
import shutil
import unicodedata
import subprocess
import numpy as np
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
from collections import Counter
import librosa

import torch
import torchaudio

# ## 3. Defining the pipeline configuration
# (dataclasses that control every aspect of the ASR pipeline)


@dataclass
class AudioConfig:
    sample_rate: int = 16000
    use_demucs: bool = False
    demucs_model: str = "htdemucs"
    denoise: bool = True
    denoise_stationary: bool = True
    denoise_prop_decrease: float = 0.75
    denoise_n_fft: int = 1024
    denoise_hop_length: int = 256


@dataclass
class VADConfig:
    threshold: float = 0.40
    min_speech_duration_ms: int = 250
    min_silence_duration_ms: int = 200
    max_speech_duration_s: float = 20.0
    speech_pad_ms: int = 400
    window_size_samples: int = 512


@dataclass
class SegmentationConfig:
    target_duration_s: float = 19.0
    max_duration_s: float = 20.1   # HF pipeline handles up to 30s
    min_duration_s: float = 1.0
    overlap_s: float = 1.0
    merge_gap_s: float = 0.3


@dataclass
class StitchedAudioConfig:
    enabled: bool = False
    silence_between_segments_s: float = 0.15 
    preserve_original_gaps: bool = False       # False = use fixed silence; True = keep original gaps
    max_gap_s: float = 2.0                     # when preserve_original_gaps=True, cap gaps to this
    chunk_length_s: float = 20.0               # pipeline's internal chunking length
    stride_length_s_left: float = 4.0          # left stride for chunk overlap
    stride_length_s_right: float = 2.0         # right stride for chunk overlap

@dataclass
class WhisperHFConfig:
    model_id: str = ""
    language: str = "bn"
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    torch_dtype: str = "float16" 
    attn_implementation: str = "sdpa"  # "sdpa", "flash_attention_2", "eager"
    use_safetensors: Optional[bool] = None  # None = auto-detect from MODEL_REGISTRY

    # ── Model mode ──
    # "base"    → use model_id directly (original behavior)
    # "merged"  → use merged_model_path (LoRA already baked in)
    # "adapter" → load base model_id + LoRA adapter from adapter_path
    # "faster"  → use faster-whisper (CTranslate2 backend) for fastest inference
    model_mode: str = "base"                    # "base", "merged", "adapter", "faster"
    merged_model_path: str = ""                 # path to merged model dir
    adapter_path: str = ""                      # path to LoRA adapter dir

    # ── Faster Whisper (CTranslate2) settings ──
    faster_model_path: str = ""                 # path to CT2-converted model dir
    faster_compute_type: str = "float16"        # "float16", "int8", "int8_float16", etc.
    faster_vad_filter: bool = False             # use faster-whisper's built-in VAD (we have our own)
    faster_no_speech_threshold: float = 0.9
    faster_log_prob_threshold: float = -2.0
    faster_compression_ratio_threshold: float = 3.0
    faster_word_timestamps: bool = True
    faster_num_gpus: int = 1                    # GPUs for parallel inference (2 for 2×T4)
    faster_batch_size: int = 16                 # batch_size for BatchedInferencePipeline

    # Decoding
    beam_size: int = 5
    return_timestamps: bool = True
    condition_on_previous_text: bool = False
    chunk_length_s: float = 20.1

    # Hallucination detection
    max_repeat_ngram: int = 4
    hallucination_min_length: int = 10  # min chars before checking


@dataclass
class PostProcessConfig:
    unicode_norm: str = "NFC"
    use_bn_unicode_normalizer: bool = True
    hallucination_filter: bool = False
    hallucination_blacklist: List[str] = field(default_factory=lambda: [])
    deloop: bool = True
    deloop_min_ngram: int = 3
    deloop_max_ngram: int = 8
    deloop_max_repeat: int = 2
    remove_english: bool = False
    strip_punctuation: bool = True
    remove_consecutive_duplicates: bool = True
    normalize_bengali_numbers: bool = False
    clean_zero_width: bool = True


@dataclass
class PipelineConfig:
    audio: AudioConfig = field(default_factory=AudioConfig)
    vad: VADConfig = field(default_factory=VADConfig)
    segmentation: SegmentationConfig = field(default_factory=SegmentationConfig)
    whisper: WhisperHFConfig = field(default_factory=WhisperHFConfig)
    postprocess: PostProcessConfig = field(default_factory=PostProcessConfig)
    stitched: StitchedAudioConfig = field(default_factory=StitchedAudioConfig)

    # Paths (customizable)
    base_data_dir: str = "/kaggle/input/competitions/dl-sprint-4-0-bengali-long-form-speech-recognition/transcription/transcription"
    train_audio_dir: str = ""    # auto-derived if empty
    train_annotation_dir: str = ""
    test_audio_dir: str = ""
    sample_submission_path: str = ""

    def __post_init__(self):
        base = Path(self.base_data_dir)
        if not self.train_audio_dir:
            self.train_audio_dir = str(base / "train" / "audio")
        if not self.train_annotation_dir:
            self.train_annotation_dir = str(base / "train" / "annotation")
        if not self.test_audio_dir:
            self.test_audio_dir = str(base / "test" / "audio")
        if not self.sample_submission_path:
            # Check common locations
            for candidate in [
                base / "sample_submission.csv",
                base / "sample_submission_asr.csv",
                Path("./bengali_long_form_asr/sample_submission_asr.csv"),
            ]:
                if candidate.exists():
                    self.sample_submission_path = str(candidate)
                    break


config = PipelineConfig()
print(f"[Config] Base data dir: {config.base_data_dir}")
print(f"[Config] Train audio: {config.train_audio_dir}")
print(f"[Config] Test audio: {config.test_audio_dir}")
print(f"[Config] Model mode: {config.whisper.model_mode}")
print(f"[Config] Model: {config.whisper.model_id}")
if config.whisper.model_mode == "merged":
    print(f"[Config] Merged path: {config.whisper.merged_model_path}")
elif config.whisper.model_mode == "adapter":
    print(f"[Config] Adapter path: {config.whisper.adapter_path}")
elif config.whisper.model_mode == "faster":
    print(f"[Config] Faster Whisper CT2 path: {config.whisper.faster_model_path}")
    print(f"[Config] Compute type: {config.whisper.faster_compute_type}")
print(f"[Config] Device: {config.whisper.device}")

# ## 3. Audio Loading & all the Preprocessings
# 


def load_audio(path: str, sr: int = 16000) -> Tuple[np.ndarray, int]:
    """Load audio file, convert to mono float32, resample to target sr."""
    waveform, orig_sr = torchaudio.load(path)

    # Convert to mono
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    # Resample if needed
    if orig_sr != sr:
        resampler = torchaudio.transforms.Resample(orig_sr, sr)
        waveform = resampler(waveform)

    audio = waveform.squeeze().numpy().astype(np.float32)
    return audio, sr


def separate_vocals(
    audio_path: str,
    output_dir: str = "./temp_demucs",
    device: str = "cuda",
) -> str:
    """Use Demucs to separate vocals. Returns path to vocals-only wav."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    stem_name = Path(audio_path).stem
    vocals_path = output_dir / "htdemucs" / stem_name / "vocals.wav"

    # Skip if already done
    if vocals_path.exists():
        print(f"[Demucs] Cached: {vocals_path}")
        return str(vocals_path)

    print(f"[Demucs] Separating vocals from {Path(audio_path).name}...")
    t0 = time.time()

    cmd = [
        "python", "-m", "demucs.separate",
        "-n", "htdemucs",
        "--two-stems", "vocals",
        "-d", device,
        "-o", str(output_dir),
        str(audio_path),
    ]
    subprocess.run(cmd, check=True, capture_output=True)

    if not vocals_path.exists():
        raise FileNotFoundError(f"Demucs output not found: {vocals_path}")

    print(f"[Demucs] Done in {time.time() - t0:.1f}s → {vocals_path}")
    return str(vocals_path)


def _cleanup_demucs_cache(audio_path: str, output_dir: str = "./temp_demucs"):
    """Remove Demucs output for a specific audio file to free disk space.
    
    Deletes the entire stem folder (e.g. temp_demucs/htdemucs/<stem>/) which
    contains both vocals.wav and no_vocals.wav (~2x original audio size).
    """
    stem_name = Path(audio_path).stem
    stem_dir = Path(output_dir) / "htdemucs" / stem_name
    if stem_dir.exists():
        try:
            total_size = sum(f.stat().st_size for f in stem_dir.rglob("*") if f.is_file())
            shutil.rmtree(stem_dir)
            print(f"[Cleanup] Removed Demucs cache: {stem_dir} ({total_size / 1e6:.1f} MB freed)")
        except Exception as e:
            print(f"[Cleanup] WARN: Failed to remove Demucs cache {stem_dir}: {e}")


def _enforce_stitched_cache_limit(
    output_dir: str = "output",
    max_size_bytes: int = 5 * 1024 * 1024 * 1024,  # 5 GB
):
    """Keep stitched audio cache under max_size_bytes by removing oldest files.
    
    Scans `output_dir` for stitched_*.wav files, and if total size exceeds
    the limit, deletes oldest files (by modification time) until under budget.
    """
    out = Path(output_dir)
    if not out.exists():
        return
    
    stitched_files = sorted(
        out.glob("stitched_*.wav"),
        key=lambda f: f.stat().st_mtime,  # oldest first
    )
    
    if not stitched_files:
        return
    
    total_size = sum(f.stat().st_size for f in stitched_files)
    
    if total_size <= max_size_bytes:
        return
    
    print(f"[Cleanup] Stitched cache: {total_size / 1e9:.2f} GB > {max_size_bytes / 1e9:.1f} GB limit")
    removed = 0
    for f in stitched_files:
        if total_size <= max_size_bytes:
            break
        fsize = f.stat().st_size
        try:
            f.unlink()
            total_size -= fsize
            removed += 1
        except Exception as e:
            print(f"[Cleanup] WARN: Could not remove {f.name}: {e}")
    
    print(f"[Cleanup] Removed {removed} old stitched files, cache now {total_size / 1e9:.2f} GB")


def smart_denoise(audio, sr=16000):
    """Adaptive denoising: only denoise if SNR is low."""
    # Estimate SNR
    rms = np.sqrt(np.mean(audio**2))
    noise_estimate = np.percentile(np.abs(audio), 10)
    snr_estimate = 20 * np.log10(rms / (noise_estimate + 1e-8))
    
    if snr_estimate < 20:  # Only denoise noisy audio
        return nr.reduce_noise(
            y=audio, sr=sr,
            stationary=False,       # non-stationary for music/speech mix
            prop_decrease=0.85,      # gentler (was 0.75)
            n_fft=2048,             # larger window for better frequency resolution
            hop_length=512,
        )
    return audio

# 2. High-pass filter to remove low-frequency rumble
from scipy.signal import butter, sosfilt

def highpass_filter(audio, sr=16000, cutoff=80):
    """Remove sub-80Hz noise (room rumble, HVAC, etc.)."""
    sos = butter(5, cutoff, btype='high', fs=sr, output='sos')
    return sosfilt(sos, audio).astype(np.float32)

# 3. Audio normalization (peak normalization)
def normalize_audio(audio, target_peak=0.95):
    """Peak-normalize to prevent clipping and ensure consistent volume."""
    peak = np.max(np.abs(audio))
    if peak > 0:
        return audio * (target_peak / peak)
    return audio



def denoise_audio(
    audio: np.ndarray,
    sr: int = 16000,
    stationary: bool = True,
    prop_decrease: float = 0.75,
    n_fft: int = 1024,
    hop_length: int = 256,
) -> np.ndarray:
    """Spectral gating noise reduction via noisereduce."""
    import noisereduce as nr

    print(f"[Denoise] Applying spectral gating...")
    t0 = time.time()
    # cleaned = nr.reduce_noise(
    #     y=audio, sr=sr,
    #     stationary=stationary,
    #     prop_decrease=prop_decrease,
    #     n_fft=n_fft, hop_length=hop_length,
    # )

    audio = smart_denoise(audio)
    # audio = highpass_filter(audio)
    # cleaned = normalize_audio(audio)
    
    print(f"[Denoise] Done in {time.time() - t0:.1f}s")
    return audio.astype(np.float32)

# ## 4. Detecting  speech regions using Silero VAD
# 


_vad_model = None

def _get_vad_model():
    """Lazy-load and cache Silero VAD model."""
    global _vad_model
    if _vad_model is None:
        from silero_vad import load_silero_vad
        _vad_model = load_silero_vad()
    return _vad_model


def run_vad(
    audio: np.ndarray,
    sr: int = 16000,
    threshold: float = 0.40,
    min_speech_duration_ms: int = 250,
    min_silence_duration_ms: int = 200,
    max_speech_duration_s: float = 30.0,
    speech_pad_ms: int = 400,
    window_size_samples: int = 512,
) -> List[Dict[str, float]]:
    """Run Silero VAD. Returns list of {'start': s, 'end': s} in seconds."""
    from silero_vad import get_speech_timestamps

    print(f"[VAD] Running (threshold={threshold}, pad={speech_pad_ms}ms)...")
    t0 = time.time()
    model = _get_vad_model()

    audio_tensor = torch.from_numpy(audio).float()

    speech_timestamps = get_speech_timestamps(
        audio_tensor, model,
        threshold=threshold,
        min_speech_duration_ms=min_speech_duration_ms,
        min_silence_duration_ms=min_silence_duration_ms,
        max_speech_duration_s=max_speech_duration_s,
        speech_pad_ms=speech_pad_ms,
        window_size_samples=window_size_samples,
        sampling_rate=sr,
        return_seconds=True,
    )

    speech_dur = sum(ts["end"] - ts["start"] for ts in speech_timestamps)
    total_dur = len(audio) / sr
    pct = 100 * speech_dur / total_dur if total_dur > 0 else 0

    print(
        f"[VAD] {len(speech_timestamps)} regions "
        f"({speech_dur:.1f}s / {total_dur:.1f}s = {pct:.1f}% speech) "
        f"in {time.time() - t0:.1f}s"
    )
    return speech_timestamps


# ## 5. Multi-Stage Segmentation strategy 
# (to create Whisper-ready audio chunks from VAD regions)


@dataclass
class SpeechSegment:
    """A chunk of audio ready for Whisper inference."""
    audio: np.ndarray
    start_time: float
    end_time: float
    index: int
    num_vad_regions: int = 0

    @property
    def duration(self) -> float:
        return self.end_time - self.start_time


def _merge_close_regions(
    timestamps: List[Dict[str, float]],
    merge_gap_s: float,
    max_duration_s: float = 29.0,
) -> List[Dict[str, float]]:
    """Merge VAD regions closer than merge_gap_s, respecting max duration."""
    if not timestamps:
        return []
    merged = [timestamps[0].copy()]
    for ts in timestamps[1:]:
        gap = ts["start"] - merged[-1]["end"]
        proposed_end = max(merged[-1]["end"], ts["end"])
        proposed_dur = proposed_end - merged[-1]["start"]
        if gap <= merge_gap_s and proposed_dur <= max_duration_s:
            merged[-1]["end"] = proposed_end
        else:
            merged.append(ts.copy())
    return merged


def _split_at_best_gaps(
    block_start: float, block_end: float,
    sub_regions: List[Dict[str, float]],
    target_s: float, max_s: float,
) -> List[Dict[str, float]]:
    """Split oversized block at longest internal silence gaps."""
    if not sub_regions or len(sub_regions) < 2:
        result = []
        pos = block_start
        while pos < block_end:
            chunk_end = min(pos + target_s, block_end)
            result.append({"start": pos, "end": chunk_end})
            pos = chunk_end
        return result

    gaps = []
    for i in range(len(sub_regions) - 1):
        gap_start = sub_regions[i]["end"]
        gap_end = sub_regions[i + 1]["start"]
        gap_dur = gap_end - gap_start
        if gap_dur > 0:
            gaps.append({
                "position": (gap_start + gap_end) / 2,
                "duration": gap_dur,
            })

    gaps.sort(key=lambda g: g["duration"], reverse=True)
    pieces = [{"start": block_start, "end": block_end}]

    for gap in gaps:
        new_pieces = []
        for piece in pieces:
            piece_dur = piece["end"] - piece["start"]
            if piece_dur > max_s and piece["start"] < gap["position"] < piece["end"]:
                new_pieces.append({"start": piece["start"], "end": gap["position"]})
                new_pieces.append({"start": gap["position"], "end": piece["end"]})
            else:
                new_pieces.append(piece)
        pieces = new_pieces
        if all(p["end"] - p["start"] <= max_s for p in pieces):
            break

    final = []
    for piece in pieces:
        dur = piece["end"] - piece["start"]
        if dur > max_s:
            pos = piece["start"]
            while pos < piece["end"]:
                chunk_end = min(pos + target_s, piece["end"])
                final.append({"start": pos, "end": chunk_end})
                pos = chunk_end
        else:
            final.append(piece)
    return final


def _pack_with_context(
    blocks: List[Dict[str, float]],
    target_s: float, max_s: float,
) -> List[Tuple[float, float, int]]:
    """Pack speech blocks into chunks with natural silence context."""
    if not blocks:
        return []
    chunks = []
    chunk_start = blocks[0]["start"]
    chunk_end = blocks[0]["end"]
    chunk_regions = 1

    for i in range(1, len(blocks)):
        block = blocks[i]
        gap = block["start"] - chunk_end
        proposed_dur = block["end"] - chunk_start
        if proposed_dur <= max_s and gap <= 10.0:
            chunk_end = block["end"]
            chunk_regions += 1
        else:
            chunks.append((chunk_start, chunk_end, chunk_regions))
            chunk_start = block["start"]
            chunk_end = block["end"]
            chunk_regions = 1

    chunks.append((chunk_start, chunk_end, chunk_regions))
    return chunks


def create_segments(
    audio: np.ndarray,
    vad_timestamps: List[Dict[str, float]],
    sr: int = 16000,
    target_duration_s: float = 15.0,
    max_duration_s: float = 28.0,
    min_duration_s: float = 1.0,
    overlap_s: float = 1.0,
    merge_gap_s: float = 0.3,
) -> List[SpeechSegment]:
    """Create Whisper-ready segments from VAD timestamps using multi-stage strategy."""
    if not vad_timestamps:
        return []

    print(
        f"[Seg] Creating segments from {len(vad_timestamps)} VAD regions "
        f"(target={target_duration_s}s, max={max_duration_s}s)..."
    )
    t0 = time.time()

    # Stage 1: Hierarchical merge
    merged = _merge_close_regions(vad_timestamps, merge_gap_s, max_duration_s)
    merged = _merge_close_regions(merged, 2.0, max_duration_s)
    merged = _merge_close_regions(merged, 5.0, max_duration_s)

    print(f"[Seg]   Merged {len(vad_timestamps)} → {len(merged)} blocks")

    # Stage 2: Split oversized
    split_blocks = []
    for block in merged:
        dur = block["end"] - block["start"]
        if dur > max_duration_s:
            sub_regions = [
                ts for ts in vad_timestamps
                if ts["start"] >= block["start"] - 0.1 and ts["end"] <= block["end"] + 0.1
            ]
            split_blocks.extend(
                _split_at_best_gaps(block["start"], block["end"], sub_regions, target_duration_s, max_duration_s)
            )
        else:
            split_blocks.append(block)

    # Stage 3: Bin-packing with natural context
    chunks = _pack_with_context(split_blocks, target_duration_s, max_duration_s)

    # Stage 4: Create SpeechSegment objects
    total_audio_duration = len(audio) / sr
    segments = []

    for idx, (start, end, n_regions) in enumerate(chunks):
        start = max(0, min(start, total_audio_duration))
        end = max(0, min(end, total_audio_duration))
        dur = end - start
        if dur < min_duration_s:
            continue

        actual_start = max(0, start - overlap_s) if idx > 0 else start
        actual_end = min(total_audio_duration, end + overlap_s) if idx < len(chunks) - 1 else end

        if actual_end - actual_start > max_duration_s + 1.0:
            actual_end = actual_start + max_duration_s

        actual_start = max(0, actual_start)
        actual_end = min(total_audio_duration, actual_end)
        if actual_end <= actual_start:
            continue

        start_sample = int(actual_start * sr)
        end_sample = min(int(actual_end * sr), len(audio))
        chunk_audio = audio[start_sample:end_sample].copy()

        segments.append(SpeechSegment(
            audio=chunk_audio,
            start_time=actual_start,
            end_time=actual_end,
            index=idx,
            num_vad_regions=n_regions,
        ))

    if segments:
        durations = [s.duration for s in segments]
        print(
            f"[Seg] Created {len(segments)} segments | "
            f"Avg: {np.mean(durations):.1f}s | "
            f"Range: {np.min(durations):.1f}s–{np.max(durations):.1f}s | "
            f"{time.time() - t0:.2f}s"
        )
    else:
        print("[Seg] No segments created!")

    return segments


# ## 6. Stitching all VAD speech regions into one contiguous audio stream 
# 


def stitch_vad_audio(
    audio: np.ndarray,
    vad_timestamps: List[Dict[str, float]],
    sr: int = 16000,
    cfg: 'StitchedAudioConfig' = None,
) -> np.ndarray:
    if cfg is None:
        cfg = StitchedAudioConfig()
    
    if not vad_timestamps:
        return np.array([], dtype=np.float32)
    
    t0 = time.time()
    total_audio_len = len(audio) / sr
    
    # Sort by start time (should already be, but be safe)
    sorted_ts = sorted(vad_timestamps, key=lambda x: x['start'])
    
    pieces = []
    fixed_silence = np.zeros(int(cfg.silence_between_segments_s * sr), dtype=np.float32)
    
    for i, ts in enumerate(sorted_ts):
        start_sample = max(0, int(ts['start'] * sr))
        end_sample = min(len(audio), int(ts['end'] * sr))
        
        if end_sample <= start_sample:
            continue
        
        # Add silence gap before this region (skip for first region)
        if i > 0 and len(pieces) > 0:
            if cfg.preserve_original_gaps:
                # Keep original gap, but cap at max_gap_s
                original_gap = ts['start'] - sorted_ts[i - 1]['end']
                gap_duration = min(original_gap, cfg.max_gap_s)
                gap_duration = max(gap_duration, 0.05)  # at least 50ms
                gap_samples = int(gap_duration * sr)
                
                # Use actual audio from the gap region (preserves ambient/room tone)
                gap_start = int(sorted_ts[i - 1]['end'] * sr)
                gap_end = min(gap_start + gap_samples, start_sample)
                if gap_end > gap_start:
                    pieces.append(audio[gap_start:gap_end].copy())
                else:
                    pieces.append(np.zeros(gap_samples, dtype=np.float32))
            else:
                # Fixed short silence
                pieces.append(fixed_silence.copy())
        
        # Add the speech region
        pieces.append(audio[start_sample:end_sample].copy())
    
    if not pieces:
        return np.array([], dtype=np.float32)
    
    stitched = np.concatenate(pieces)
    
    original_speech_dur = sum(ts['end'] - ts['start'] for ts in sorted_ts)
    stitched_dur = len(stitched) / sr
    compression = (1 - stitched_dur / total_audio_len) * 100 if total_audio_len > 0 else 0
    
    print(
        f"[Stitch] {len(sorted_ts)} VAD regions → {stitched_dur:.1f}s "
        f"(speech: {original_speech_dur:.1f}s, "
        f"original: {total_audio_len:.1f}s, "
        f"compressed {compression:.0f}%) "
        f"in {time.time() - t0:.2f}s"
    )
    
    return stitched


# ## 7. Loading and running the Whisper model via HuggingFace 


def _patch_generation_config(model, processor, cfg: WhisperHFConfig):
    """
    Fix generation config for all model modes (base/merged/adapter).
    
    Addresses three issues with PEFT-wrapped Whisper models:
    1. lang_to_id is None on generation_config → crash in _retrieve_init_tokens
    2. suppress_tokens on model.config → ValueError in newer transformers
       ("modified pretrained model configuration to control generation")
    3. forced_decoder_ids conflicts with language/task params
    
    Strategy:
    - Load the ORIGINAL generation_config from base model path (has correct lang_to_id)
    - Set all generation params ONLY on model.generation_config
    - DELETE generation-related attrs from model.config (newer transformers rejects them)
    """
    from transformers import GenerationConfig

    gc = model.generation_config

    # ── Step 1: Source lang_to_id from original base model's generation_config ──
    # PEFT wrapping can lose lang_to_id. Load it fresh from disk.
    if not getattr(gc, "lang_to_id", None) or not isinstance(gc.lang_to_id, dict):
        try:
            # Load the original generation_config.json which has the full lang_to_id dict
            base_path = cfg.model_id  # works for both local path and HF hub id
            if cfg.model_mode == "merged":
                base_path = cfg.merged_model_path
            base_gc = GenerationConfig.from_pretrained(base_path)
            if getattr(base_gc, "lang_to_id", None) and isinstance(base_gc.lang_to_id, dict):
                gc.lang_to_id = base_gc.lang_to_id
                print(f"[GenConfig] Loaded lang_to_id from {base_path}: {len(gc.lang_to_id)} languages")
            if getattr(base_gc, "task_to_id", None) and not getattr(gc, "task_to_id", None):
                gc.task_to_id = base_gc.task_to_id
        except Exception as e:
            print(f"[GenConfig] WARN: Could not load base generation_config: {e}")

    # If still missing, build from tokenizer (fallback)
    if not getattr(gc, "lang_to_id", None) or not isinstance(gc.lang_to_id, dict):
        print("[GenConfig] Building lang_to_id from tokenizer...")
        tokenizer = processor.tokenizer
        lang_to_id = {}
        for token, idx in tokenizer.get_vocab().items():
            if token.startswith("<|") and token.endswith("|>") and len(token) > 4:
                lang_name = token[2:-2]
                if lang_name not in ("transcribe", "translate", "startoflm", "endoftext",
                                      "startoftranscript", "notimestamps", "nospeech"):
                    lang_to_id[lang_name] = idx
        if lang_to_id:
            gc.lang_to_id = lang_to_id
            print(f"[GenConfig] Built lang_to_id: {len(lang_to_id)} languages")

    if not getattr(gc, "task_to_id", None):
        gc.task_to_id = {
            "transcribe": processor.tokenizer.convert_tokens_to_ids("<|transcribe|>"),
            "translate": processor.tokenizer.convert_tokens_to_ids("<|translate|>"),
        }

    # ── Step 2: Set generation params on generation_config ONLY ──
    if gc.suppress_tokens is None:
        gc.suppress_tokens = []
    gc.forced_decoder_ids = None
    gc.language = cfg.language
    gc.task = "transcribe"
    gc.num_beams = cfg.beam_size

    # ── Step 3: DELETE generation attrs from model.config ──
    # Newer transformers raises ValueError if model.config has generation params
    # that differ from defaults. Must remove them entirely.
    configs_to_clean = [model.config]
    if hasattr(model, "base_model") and hasattr(model.base_model, "model"):
        configs_to_clean.append(model.base_model.model.config)

    for mc in configs_to_clean:
        for attr in ("suppress_tokens", "forced_decoder_ids", "begin_suppress_tokens"):
            if hasattr(mc, attr):
                try:
                    delattr(mc, attr)
                except (AttributeError, TypeError):
                    try:
                        setattr(mc, attr, None)
                    except Exception:
                        pass

    # ── Step 4: Patch inner model generation_config for PEFT ──
    # PEFT delegates generate() to base_model.model — its gen_config must also be correct
    if hasattr(model, "base_model") and hasattr(model.base_model, "model"):
        inner = model.base_model.model
        if hasattr(inner, "generation_config"):
            igc = inner.generation_config
            if igc.suppress_tokens is None:
                igc.suppress_tokens = []
            igc.forced_decoder_ids = None
            igc.language = cfg.language
            igc.task = "transcribe"
            igc.num_beams = cfg.beam_size
            if not getattr(igc, "lang_to_id", None) or not isinstance(igc.lang_to_id, dict):
                igc.lang_to_id = gc.lang_to_id
            if not getattr(igc, "task_to_id", None):
                igc.task_to_id = gc.task_to_id

    # Enable KV cache for inference speed
    if hasattr(model, "base_model") and hasattr(model.base_model, "model"):
        model.base_model.model.config.use_cache = True
    else:
        model.config.use_cache = True


_whisper_pipe = None
_whisper_model_id = None
_whisper_model_mode = None


def _get_whisper_pipe(cfg: WhisperHFConfig):
    """
    Lazy-load and cache the HF Whisper pipeline.
    Supports three model modes:
      - "base":    Load model_id directly (original behavior)
      - "merged":  Load from merged_model_path (LoRA already baked in)
      - "adapter": Load base model_id + apply LoRA adapter from adapter_path
    """
    global _whisper_pipe, _whisper_model_id, _whisper_model_mode

    # Cache key includes both model identity and mode
    cache_key = f"{cfg.model_mode}:{cfg.model_id}:{cfg.merged_model_path}:{cfg.adapter_path}"
    if _whisper_pipe is not None and _whisper_model_id == cache_key:
        return _whisper_pipe

    from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

    device = cfg.device
    torch_dtype = torch.float16 if cfg.torch_dtype == "float16" else torch.bfloat16 if cfg.torch_dtype == "bfloat16" else torch.float32

    mode = cfg.model_mode
    t0 = time.time()

    if mode == "merged":
        # ── Merged model: LoRA weights already baked into base ──
        model_path = cfg.merged_model_path
        if not model_path:
            raise ValueError("merged_model_path must be set when model_mode='merged'")
        print(f"[Whisper] Loading MERGED model from {model_path}...")

        model = AutoModelForSpeechSeq2Seq.from_pretrained(
            model_path,
            torch_dtype=torch_dtype,
            attn_implementation=cfg.attn_implementation,
        ).to(device)

        processor = AutoProcessor.from_pretrained(model_path)

    elif mode == "adapter":
        # ── Adapter mode: load base + apply LoRA ──
        if not cfg.adapter_path:
            raise ValueError("adapter_path must be set when model_mode='adapter'")
        print(f"[Whisper] Loading BASE model: {cfg.model_id}")
        print(f"[Whisper] Applying LoRA adapter from: {cfg.adapter_path}")

        from peft import PeftModel

        base_model = AutoModelForSpeechSeq2Seq.from_pretrained(
            cfg.model_id,
            torch_dtype=torch_dtype,
            attn_implementation=cfg.attn_implementation,
        )
        model = PeftModel.from_pretrained(base_model, cfg.adapter_path)
        model = model.to(device)
        model.eval()

        # Processor from adapter dir (has tokenizer config) or fall back to base
        try:
            processor = AutoProcessor.from_pretrained(cfg.adapter_path)
        except Exception:
            print("Fallback to base")
            processor = AutoProcessor.from_pretrained(cfg.model_id)

    else:
        # ── Base mode: original behavior ──
        print(f"[Whisper] Loading BASE model: {cfg.model_id}...")

        model = AutoModelForSpeechSeq2Seq.from_pretrained(
            cfg.model_id,
            torch_dtype=torch_dtype,
            attn_implementation=cfg.attn_implementation,
        ).to(device)

        processor = AutoProcessor.from_pretrained(cfg.model_id)

    # ── Patch generation config (fixes suppress_tokens=None crash) ──
    _patch_generation_config(model, processor, cfg)

    # ── DEBUG: Verify patch worked ──
    print(f"[DEBUG] model type: {type(model).__name__}")
    gc = model.generation_config
    print(f"[DEBUG] generation_config.suppress_tokens: {type(gc.suppress_tokens).__name__} (len={len(gc.suppress_tokens) if gc.suppress_tokens else 'N/A'})")
    print(f"[DEBUG] generation_config.lang_to_id: {type(getattr(gc, 'lang_to_id', None)).__name__} (len={len(gc.lang_to_id) if isinstance(getattr(gc, 'lang_to_id', None), dict) else 'N/A'})")
    print(f"[DEBUG] generation_config.task_to_id: {getattr(gc, 'task_to_id', None)}")
    print(f"[DEBUG] generation_config.language={gc.language}, task={gc.task}, num_beams={gc.num_beams}")
    print(f"[DEBUG] model.config has suppress_tokens: {hasattr(model.config, 'suppress_tokens')}")
    if hasattr(model, 'base_model') and hasattr(model.base_model, 'model'):
        inner = model.base_model.model
        igc = inner.generation_config
        print(f"[DEBUG] INNER generation_config.lang_to_id: {type(getattr(igc, 'lang_to_id', None)).__name__} (len={len(igc.lang_to_id) if isinstance(getattr(igc, 'lang_to_id', None), dict) else 'N/A'})")
        print(f"[DEBUG] INNER config has suppress_tokens: {hasattr(inner.config, 'suppress_tokens')}")

    # ── Build pipeline ──
    # NOTE: Do NOT pass generate_kwargs here — all generation params are baked
    # into model.generation_config above. Passing them in both places triggers
    # "generation_config together with generation-related arguments" deprecation
    # and can cause param conflicts.
    pipe = pipeline(
        task="automatic-speech-recognition",
        model=model,
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        device=device,
        chunk_length_s=20.1,
        return_timestamps=False,
        dtype=torch_dtype,           # was torch_dtype= (deprecated in newer transformers)
        ignore_warning=True,         # suppress seq2seq chunk_length_s warning
    )

    print(f"[Whisper] Loaded [{mode.upper()}] in {time.time() - t0:.1f}s | dtype={torch_dtype}")

    _whisper_pipe = pipe
    _whisper_model_id = cache_key
    _whisper_model_mode = mode
    return pipe


# ── Faster Whisper (CTranslate2) multi-GPU model pool ──────────────────────

from concurrent.futures import ThreadPoolExecutor, as_completed

_faster_whisper_models = {}       # {gpu_index: BatchedInferencePipeline}
_faster_whisper_raw_models = {}   # {gpu_index: WhisperModel} (kept for ref)
_faster_whisper_pool_path = None


def _get_faster_whisper_models(cfg: WhisperHFConfig):
    """
    Lazy-load and cache BatchedInferencePipeline on each GPU.
    Returns list of (gpu_index, batched_pipeline) tuples.
    """
    global _faster_whisper_models, _faster_whisper_raw_models, _faster_whisper_pool_path

    num_gpus = cfg.faster_num_gpus
    model_path = cfg.faster_model_path
    if not model_path:
        raise ValueError("faster_model_path must be set when model_mode='faster'")

    # Return cached if same config
    if _faster_whisper_models and _faster_whisper_pool_path == model_path and len(_faster_whisper_models) == num_gpus:
        return list(_faster_whisper_models.items())

    from faster_whisper import WhisperModel, BatchedInferencePipeline

    _faster_whisper_models.clear()
    _faster_whisper_raw_models.clear()

    t0 = time.time()
    actual_gpus = min(num_gpus, torch.cuda.device_count()) if torch.cuda.is_available() else 1
    print(f"[Whisper-Faster] Loading CTranslate2 model on {actual_gpus} GPU(s)...")

    for gpu_id in range(actual_gpus):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        gt0 = time.time()
        raw_model = WhisperModel(
            model_path,
            device=device,
            device_index=gpu_id,
            compute_type=cfg.faster_compute_type,
        )
        batched = BatchedInferencePipeline(model=raw_model)
        _faster_whisper_raw_models[gpu_id] = raw_model
        _faster_whisper_models[gpu_id] = batched
        print(f"  GPU {gpu_id}: loaded in {time.time() - gt0:.1f}s")

    _faster_whisper_pool_path = model_path
    print(f"[Whisper-Faster] All {actual_gpus} GPU(s) ready in {time.time() - t0:.1f}s | compute_type={cfg.faster_compute_type}")
    return list(_faster_whisper_models.items())


def _get_faster_whisper_model(cfg: WhisperHFConfig):
    """Backward-compatible: returns the GPU 0 batched pipeline."""
    models = _get_faster_whisper_models(cfg)
    return models[0][1]  # (gpu_id, pipeline) → pipeline


def _is_hallucination(text: str, max_repeat_ngram: int = 4, min_length: int = 10) -> bool:
    """Detect hallucinated output via repeated n-gram patterns."""
    text = text.strip()
    if not text or len(text) < min_length:
        return False

    words = text.split()
    if len(words) >= max_repeat_ngram * 2:
        for n in range(2, min(6, len(words) // 2 + 1)):
            ngrams = [" ".join(words[i:i + n]) for i in range(len(words) - n + 1)]
            counts = Counter(ngrams)
            if counts and counts.most_common(1)[0][1] > max_repeat_ngram:
                return True
    return False


@dataclass
class SegmentResult:
    """Result from transcribing a single segment."""
    text: str
    start_time: float
    end_time: float
    is_hallucination: bool = False


def transcribe_segments(
    segments: List[SpeechSegment],
    cfg: WhisperHFConfig,
    verbose: bool = True,
) -> Tuple[str, List[SegmentResult]]:
    """
    Transcribe a list of speech segments using HF Whisper pipeline.
    Returns (full_text, list_of_segment_results).
    """
    pipe = _get_whisper_pipe(cfg)
    t0 = time.time()
    all_results = []
    all_texts = []
    hallucination_count = 0

    # NOTE: generation params (language, num_beams, etc.) are baked into
    # model.generation_config by _patch_generation_config — no need to
    # pass generate_kwargs here.

    # ── DEBUG: Test model.generate() directly on first segment ──
    if segments:
        _test_seg = segments[0]
        _test_feats = pipe.feature_extractor(
            _test_seg.audio, sampling_rate=16000, return_tensors="pt"
        ).input_features.to(pipe.device)
        try:
            with torch.no_grad():
                _test_ids = pipe.model.generate(
                    _test_feats,
                    language=cfg.language,
                    task="transcribe",
                    num_beams=4,  # greedy for speed beam_size
                    max_new_tokens=10,
                )
            _test_text = pipe.tokenizer.decode(_test_ids[0], skip_special_tokens=True)
            print(f"[DEBUG] Direct model.generate() OK: '{_test_text[:60]}'")
        except Exception as e:
            import traceback
            print(f"[DEBUG] Direct model.generate() FAILED: {e}")
            traceback.print_exc()

    for i, seg in enumerate(segments):
        if verbose:
            print(
                f"  [{i + 1}/{len(segments)}] "
                f"{seg.start_time:.1f}s–{seg.end_time:.1f}s "
                f"({seg.duration:.1f}s)...",
                end=" ", flush=True,
            )

        try:
            # HF pipeline accepts numpy array with sampling_rate (librosa-style)
            result = pipe(
                {"array": seg.audio, "sampling_rate": 16000},
                batch_size=16,
            )
            text = result.get("text", "").strip()
        except Exception as e:
            if i < 2:  # Full traceback for first 2 failures
                import traceback
                print(f"✗ Error: {e}")
                traceback.print_exc()
            else:
                print(f"✗ Error: {e}")
            text = ""

        seg_result = SegmentResult(
            text=text,
            start_time=seg.start_time,
            end_time=seg.end_time,
        )

        # Hallucination check
        if _is_hallucination(text, cfg.max_repeat_ngram, cfg.hallucination_min_length):
            seg_result.is_hallucination = True
            hallucination_count += 1
            if verbose:
                print(f"⚠ HALLUCINATION")
            all_results.append(seg_result)
            continue

        if text:
            all_results.append(seg_result)
            all_texts.append(text)
            if verbose:
                preview = text[:60] + "..." if len(text) > 60 else text
                print(f"✓ {preview}")
        else:
            all_results.append(seg_result)
            if verbose:
                print("(empty)")

    full_text = " ".join(all_texts)
    elapsed = time.time() - t0

    good_count = len(all_texts)
    if verbose:
        print(
            f"\n[Whisper] Transcribed {good_count}/{len(segments)} segments, "
            f"{hallucination_count} hallucinations | "
            f"{len(full_text)} chars | {elapsed:.1f}s"
        )

    return full_text, all_results


def _pad_segment_audio(audio: np.ndarray, sr: int = 16000) -> np.ndarray:
    """Pad audio with 1s of faint white noise at start/end to prevent decoder boundary artifacts."""
    pad_samples = int(1.0 * sr)
    noise_level = 1e-4  # ~ -80dB
    pad_start = (np.random.randn(pad_samples) * noise_level).astype(np.float32)
    pad_end = (np.random.randn(pad_samples) * noise_level).astype(np.float32)
    return np.concatenate([pad_start, audio, pad_end])


def _transcribe_group_on_gpu(
    batched_pipeline, segment_group: List[Tuple[int, SpeechSegment]],
    total: int, cfg: WhisperHFConfig, gpu_id: int, verbose: bool = True,
) -> List[Tuple[int, SegmentResult, str]]:
    """
    Transcribe a group of segments SEQUENTIALLY on one GPU.
    Each GPU gets its own group and processes it one-by-one — no concurrent
    transcribe() calls on the same pipeline, avoiding CUDA conflicts.
    Returns list of (original_index, SegmentResult, text).
    """
    results = []
    for seg_idx, seg in segment_group:
        try:
            padded_audio = _pad_segment_audio(seg.audio)

            fw_segments, info = batched_pipeline.transcribe(
                padded_audio,
                batch_size=cfg.faster_batch_size,
                language=cfg.language,
                beam_size=cfg.beam_size,
                condition_on_previous_text=cfg.condition_on_previous_text,
                vad_filter=cfg.faster_vad_filter,
                word_timestamps=cfg.faster_word_timestamps,
                no_speech_threshold=cfg.faster_no_speech_threshold,
                log_prob_threshold=cfg.faster_log_prob_threshold,
                compression_ratio_threshold=cfg.faster_compression_ratio_threshold,
                without_timestamps=False,
            )
            text_parts = []
            for fw_seg in fw_segments:
                text_parts.append(fw_seg.text.strip())
            text = " ".join(text_parts).strip()
        except Exception as e:
            text = ""
            if verbose:
                print(f"  [GPU {gpu_id}] [{seg_idx + 1}/{total}] ✗ Error: {e}")

        if verbose and text:
            preview = text[:50] + "..." if len(text) > 50 else text
            print(f"  [GPU {gpu_id}] [{seg_idx + 1}/{total}] {seg.start_time:.1f}s–{seg.end_time:.1f}s ✓ {preview}")
        elif verbose:
            print(f"  [GPU {gpu_id}] [{seg_idx + 1}/{total}] {seg.start_time:.1f}s–{seg.end_time:.1f}s (empty)")

        seg_result = SegmentResult(
            text=text,
            start_time=seg.start_time,
            end_time=seg.end_time,
        )
        results.append((seg_idx, seg_result, text))

    return results


def transcribe_segments_faster(
    segments: List[SpeechSegment],
    cfg: WhisperHFConfig,
    verbose: bool = True,
) -> Tuple[str, List[SegmentResult]]:
    """
    Transcribe segments using faster-whisper BatchedInferencePipeline.
    
    Strategy: Split segments into groups (one per GPU). Each GPU processes its
    group SEQUENTIALLY in its own thread — no concurrent transcribe() calls on
    the same pipeline. The GPU threads run in PARALLEL with each other.
    This avoids CUDA concurrency errors while still utilizing both GPUs.
    
    Returns (full_text, list_of_segment_results).
    """
    gpu_models = _get_faster_whisper_models(cfg)
    num_gpus = len(gpu_models)
    t0 = time.time()

    if verbose:
        print(f"[Whisper-Faster] Transcribing {len(segments)} segments across {num_gpus} GPU(s) (batch_size={cfg.faster_batch_size})...")

    # ── Split segments into groups: one per GPU (round-robin) ──
    gpu_groups = [[] for _ in range(num_gpus)]
    for i, seg in enumerate(segments):
        gpu_groups[i % num_gpus].append((i, seg))

    if verbose:
        for g_idx, group in enumerate(gpu_groups):
            print(f"  GPU {gpu_models[g_idx][0]}: {len(group)} segments")

    # ── Each GPU processes its group sequentially, GPUs run in parallel ──
    results_map = {}  # seg_idx → (SegmentResult, text)
    hallucination_count = 0

    if num_gpus == 1:
        # Single GPU — just run directly, no threading overhead
        gpu_id, batched_pipeline = gpu_models[0]
        group_results = _transcribe_group_on_gpu(
            batched_pipeline, gpu_groups[0], len(segments), cfg, gpu_id, verbose
        )
        for seg_idx, seg_result, text in group_results:
            results_map[seg_idx] = (seg_result, text)
    else:
        # Multi-GPU — each GPU thread processes its group sequentially
        with ThreadPoolExecutor(max_workers=num_gpus) as executor:
            futures = {}
            for g_idx in range(num_gpus):
                gpu_id, batched_pipeline = gpu_models[g_idx]
                future = executor.submit(
                    _transcribe_group_on_gpu,
                    batched_pipeline, gpu_groups[g_idx], len(segments),
                    cfg, gpu_id, verbose,
                )
                futures[future] = g_idx

            for future in as_completed(futures):
                group_results = future.result()
                for seg_idx, seg_result, text in group_results:
                    results_map[seg_idx] = (seg_result, text)

    # ── Hallucination check + reassemble in original order ──
    all_results = []
    all_texts = []
    for i in range(len(segments)):
        seg_result, text = results_map[i]
        if _is_hallucination(text, cfg.max_repeat_ngram, cfg.hallucination_min_length):
            seg_result.is_hallucination = True
            hallucination_count += 1
            if verbose:
                print(f"  [{i + 1}/{len(segments)}] ⚠ HALLUCINATION")
            all_results.append(seg_result)
        else:
            all_results.append(seg_result)
            if text:
                all_texts.append(text)

    full_text = " ".join(all_texts)
    elapsed = time.time() - t0

    if verbose:
        print(
            f"\n[Whisper-Faster] Transcribed {len(all_texts)}/{len(segments)} segments "
            f"on {num_gpus} GPU(s), {hallucination_count} hallucinations | "
            f"{len(full_text)} chars | {elapsed:.1f}s"
        )

    return full_text, all_results


def transcribe_stitched(
    stitched_audio_path: str,
    whisper_cfg: WhisperHFConfig,
    stitched_cfg: 'StitchedAudioConfig' = None,
    verbose: bool = True,
) -> str:
    """Transcribe stitched audio using HF pipeline's built-in chunking.
    
    Loads the saved stitched WAV file via librosa (consistent preprocessing),
    then passes it to the pipeline with chunk_length_s, letting Whisper
    handle its own chunking and cross-chunk context.
    
    Args:
        stitched_audio_path: Path to saved stitched WAV file
        whisper_cfg: Whisper pipeline config
        stitched_cfg: Stitched mode config (for chunk_length_s, stride)
        verbose: Print progress
    
    Returns:
        Raw transcribed text (before post-processing)
    """
    if stitched_cfg is None:
        stitched_cfg = StitchedAudioConfig()
    
    pipe = _get_whisper_pipe(whisper_cfg)
    t0 = time.time()
    
    # Load via librosa for consistent preprocessing (resampling, normalization)
    audio, sr = librosa.load(stitched_audio_path, sr=16000)
    
    duration = len(audio) / sr
    if verbose:
        print(
            f"[Whisper-Stitched] Transcribing {duration:.1f}s of stitched audio "
            f"(chunk={stitched_cfg.chunk_length_s}s, "
            f"stride=({stitched_cfg.stride_length_s_left}, {stitched_cfg.stride_length_s_right})s)..."
        )
    
    # NOTE: generation params baked into model.generation_config
    
    try:
        result = pipe(
            {"array": audio, "sampling_rate": sr},
            batch_size=16
            
        )
        text = result.get("text", "").strip()
    except Exception as e:
        print(f"[Whisper-Stitched] Error: {e}")
        text = ""
    
    elapsed = time.time() - t0
    
    # Hallucination check on full output
    is_hallucinated = _is_hallucination(
        text, whisper_cfg.max_repeat_ngram, whisper_cfg.hallucination_min_length
    )
    
    if verbose:
        if is_hallucinated:
            print(f"[Whisper-Stitched] ⚠ HALLUCINATION detected in output")
        preview = text[:100] + "..." if len(text) > 100 else text
        print(
            f"[Whisper-Stitched] Done in {elapsed:.1f}s | "
            f"{len(text)} chars | RTF: {elapsed / duration:.2f}x"
        )
        if text:
            print(f"  Preview: {preview}")
    
    return text


def _transcribe_audio_chunk_on_gpu(
    batched_pipeline, audio: np.ndarray, cfg: WhisperHFConfig, gpu_id: int,
) -> str:
    """Transcribe an audio chunk on a specific GPU. Thread-safe."""
    try:
        fw_segments, info = batched_pipeline.transcribe(
            audio,
            batch_size=cfg.faster_batch_size,
            language=cfg.language,
            beam_size=cfg.beam_size,
            condition_on_previous_text=cfg.condition_on_previous_text,
            vad_filter=cfg.faster_vad_filter,
            word_timestamps=cfg.faster_word_timestamps,
            no_speech_threshold=cfg.faster_no_speech_threshold,
            log_prob_threshold=cfg.faster_log_prob_threshold,
            compression_ratio_threshold=cfg.faster_compression_ratio_threshold,
            without_timestamps=False,
        )
        text_parts = []
        for fw_seg in fw_segments:
            text_parts.append(fw_seg.text.strip())
        return " ".join(text_parts).strip()
    except Exception as e:
        print(f"[GPU {gpu_id}] Transcription error: {e}")
        return ""


def transcribe_stitched_faster(
    stitched_audio_path: str,
    whisper_cfg: WhisperHFConfig,
    stitched_cfg: 'StitchedAudioConfig' = None,
    verbose: bool = True,
) -> str:
    """Transcribe stitched audio using faster-whisper BatchedInferencePipeline.
    
    For multi-GPU: splits audio in half and transcribes on separate GPUs.
    Uses batch_size for internal chunk parallelism on each GPU.
    
    Returns:
        Raw transcribed text (before post-processing)
    """
    if stitched_cfg is None:
        stitched_cfg = StitchedAudioConfig()
    
    gpu_models = _get_faster_whisper_models(whisper_cfg)
    num_gpus = len(gpu_models)
    t0 = time.time()
    
    # Load via librosa for consistent preprocessing
    audio, sr = librosa.load(stitched_audio_path, sr=16000)
    
    duration = len(audio) / sr
    if verbose:
        print(
            f"[Whisper-Faster-Stitched] Transcribing {duration:.1f}s on {num_gpus} GPU(s) "
            f"(batch_size={whisper_cfg.faster_batch_size})..."
        )
    
    if num_gpus >= 2 and duration > 30.0:
        # ── Split audio across GPUs ──
        mid_sample = len(audio) // 2
        # Find a silence point near the midpoint to avoid cutting mid-word
        search_window = int(2.0 * sr)  # search ±2s around midpoint
        search_start = max(0, mid_sample - search_window)
        search_end = min(len(audio), mid_sample + search_window)
        window = np.abs(audio[search_start:search_end])
        # Find quietest 0.1s block
        block = int(0.1 * sr)
        if len(window) > block:
            energies = np.array([np.mean(window[j:j+block]) for j in range(0, len(window) - block, block // 2)])
            best_offset = np.argmin(energies) * (block // 2)
            split_point = search_start + best_offset
        else:
            split_point = mid_sample
        
        chunk_0 = audio[:split_point]
        chunk_1 = audio[split_point:]
        
        if verbose:
            print(f"  Split: GPU 0 → {len(chunk_0)/sr:.1f}s | GPU 1 → {len(chunk_1)/sr:.1f}s")
        
        with ThreadPoolExecutor(max_workers=num_gpus) as executor:
            f0 = executor.submit(_transcribe_audio_chunk_on_gpu, gpu_models[0][1], chunk_0, whisper_cfg, 0)
            f1 = executor.submit(_transcribe_audio_chunk_on_gpu, gpu_models[1][1], chunk_1, whisper_cfg, 1)
            text_0 = f0.result()
            text_1 = f1.result()
        
        text = (text_0 + " " + text_1).strip()
    else:
        # ── Single GPU with batching ──
        gpu_id, batched_pipeline = gpu_models[0]
        text = _transcribe_audio_chunk_on_gpu(batched_pipeline, audio, whisper_cfg, gpu_id)
    
    elapsed = time.time() - t0
    
    # Hallucination check on full output
    is_hallucinated = _is_hallucination(
        text, whisper_cfg.max_repeat_ngram, whisper_cfg.hallucination_min_length
    )
    
    if verbose:
        if is_hallucinated:
            print(f"[Whisper-Faster-Stitched] ⚠ HALLUCINATION detected in output")
        preview = text[:100] + "..." if len(text) > 100 else text
        print(
            f"[Whisper-Faster-Stitched] Done in {elapsed:.1f}s | "
            f"{len(text)} chars | RTF: {elapsed / duration:.2f}x"
        )
        if text:
            print(f"  Preview: {preview}")
    
    return text


# ## 8. Our Multi-stage Bengali text Post-processing Pipeline


# ─── Stage 1: Unicode Normalization ──────────────────────────────────────────

def normalize_unicode(text: str, form: str = "NFC") -> str:
    """NFC normalization. IMPORTANT: NOT NFKC — preserves Bengali vowel signs."""
    return unicodedata.normalize(form, text)


# ─── Stage 2: Bengali-specific Normalization ─────────────────────────────────

def normalize_bengali_unicode(text: str) -> str:
    """Apply bnUnicodeNormalizer for grapheme canonicalization."""
    try:
        from bnunicodenormalizer import Normalizer
        normalizer = Normalizer()
        result = normalizer(text)
        normalized = result["normalized"] if result["normalized"] else text
        return normalized
    except ImportError:
        print("[WARN] bnunicodenormalizer not installed, skipping")
        return text
    except Exception:
        return text


# ─── Stage 3: Zero-width Character Cleanup (Competition-standard) ──────────

_ZW_RE = re.compile(r"[\u200B-\u200D\uFEFF]")  # ZWS, ZWNJ, ZWJ, BOM

def clean_zero_width(text: str) -> str:
    """
    Remove zero-width characters and BOM using competition-standard regex.
    Also normalizes NBSP to regular space.
    """
    text = _ZW_RE.sub("", text)
    text = text.replace("\u00A0", " ")  # NBSP → space
    return text


# ─── Stage 4: Hasanta/Nukta Normalization ────────────────────────────────────

def normalize_hasanta(text: str) -> str:
    """
    Clean up hasanta (্ U+09CD) placement issues:
    - Remove double hasanta (্্ → ্)
    - Remove hasanta at word boundaries (shouldn't exist)
    - Remove orphan hasanta (not between consonants)
    """
    # Remove double hasanta
    text = re.sub(r'\u09CD{2,}', '\u09CD', text)

    # Remove hasanta at start/end of words
    text = re.sub(r'(?<=\s)\u09CD', '', text)
    text = re.sub(r'\u09CD(?=\s|$)', '', text)

    return text


# ─── Stage 5: Hallucination Blacklist ────────────────────────────────────────

def filter_hallucination_phrases(text: str, blacklist: List[str]) -> str:
    """Remove known hallucination phrases (YouTube artifacts, etc.)."""
    for phrase in blacklist:
        text = text.replace(phrase, " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


# ─── Stage 6: N-gram De-looping ─────────────────────────────────────────────

def deloop_text(
    text: str,
    min_ngram: int = 3,
    max_ngram: int = 8,
    max_repeat: int = 2,
) -> str:
    """Remove repeated n-gram sequences (Whisper hallucination pattern)."""
    words = text.split()
    if len(words) < min_ngram * 2:
        return text

    for n in range(max_ngram, min_ngram - 1, -1):
        i = 0
        new_words = []
        while i < len(words):
            if i + n * 2 <= len(words):
                ngram = words[i:i + n]
                repeat_count = 1
                j = i + n
                while j + n <= len(words) and words[j:j + n] == ngram:
                    repeat_count += 1
                    j += n

                if repeat_count > max_repeat:
                    for _ in range(min(max_repeat, 1)):
                        new_words.extend(ngram)
                    i = j
                    continue

            new_words.append(words[i])
            i += 1

        words = new_words

    return " ".join(words)


# ─── Stage 7: Consecutive Duplicate Word Removal ─────────────────────────────

def remove_consecutive_duplicates(text: str) -> str:
    """
    Remove words repeated 3+ times consecutively (Whisper stutter artifact).
    Preserves valid Bengali reduplications (e.g., 'ধীরে ধীরে', 'বার বার').
    """
    words = text.split()
    if len(words) < 2:
        return text

    result = [words[0]]
    count = 1
    for i in range(1, len(words)):
        if words[i] == result[-1]:
            count += 1
            if count < 3:  # allow up to 2 consecutive (valid reduplication)
                result.append(words[i])
        else:
            result.append(words[i])
            count = 1

    return " ".join(result)


# ─── Stage 8: Bengali Number Word Conversion ─────────────────────────────────

BENGALI_DIGIT_WORDS = {
    '০': 'শূন্য', '১': 'এক', '২': 'দুই', '৩': 'তিন', '৪': 'চার',
    '৫': 'পাঁচ', '৬': 'ছয়', '৭': 'সাত', '৮': 'আট', '৯': 'নয়',
}


def bengali_number_to_words(text: str) -> str:
    """
    Convert isolated Bengali digits to their word forms.
    Only converts single standalone digits — multi-digit numbers are left as-is
    (proper conversion would need a full number-to-word library).
    """
    for digit, word in BENGALI_DIGIT_WORDS.items():
        # Only replace standalone digits (surrounded by spaces or at boundaries)
        text = re.sub(rf'(?<!\S){re.escape(digit)}(?!\S)', word, text)
    return text


# ─── Stage 9: Bengali Character Cleanup ──────────────────────────────────────

_ASCII_TO_BENGALI_DIGITS = str.maketrans("0123456789", "০১২৩৪৫৬৭৮৯")

def clean_bengali_text(
    text: str,
    remove_english: bool = False,
    strip_punctuation: bool = True,
) -> str:
    """Clean Bengali text: punctuation, convert ASCII digits to Bengali, spacing."""
    if strip_punctuation:
        # Remove Bengali and common punctuation
        text = re.sub(r'[।,;:!?\-\—\–\"\'\(\)\[\]\{\}…\u0964\u0965]', ' ', text)
        # Remove remaining ASCII punctuation
        text = re.sub(r'[.,;:!?"\'\-\(\)\[\]\{\}/\\@#$%^&*_+=<>~`|]', ' ', text)

    if remove_english:
        text = re.sub(r'[a-zA-Z]', '', text)

    # Convert ASCII digits to Bengali digits (competition requires Bengali digits)
    text = text.translate(_ASCII_TO_BENGALI_DIGITS)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# ─── Stage 10: Orphan Diacritics Cleanup ─────────────────────────────────────

def clean_orphan_diacritics(text: str) -> str:
    """
    Remove Bengali diacritics (vowel signs, chandrabindu, visarga, anusvara)
    that appear without a preceding consonant — likely ASR artifacts.
    
    Bengali vowel signs: া ি ী ু ূ ে ৈ ো ৌ (U+09BE–U+09CC)
    Chandrabindu: ঁ (U+0981), Anusvara: ং (U+0982), Visarga: ঃ (U+0983)
    """
    # Remove vowel signs at word start (no preceding consonant)
    text = re.sub(r'(?<=\s)[\u09BE-\u09CC\u09D7]+', '', text)
    text = re.sub(r'^[\u09BE-\u09CC\u09D7]+', '', text)

    # Remove standalone chandrabindu/anusvara/visarga
    text = re.sub(r'(?<=\s)[\u0981-\u0983]+(?=\s|$)', '', text)
    text = re.sub(r'^[\u0981-\u0983]+(?=\s|$)', '', text)

    return text


# ─── Stage 11: Non-Bengali Script Removal ────────────────────────────────────

# Bengali Unicode block: U+0980–U+09FF
# Allow: Bengali chars, Bengali digits, spaces, and common Bengali punctuation (।)
_NON_BENGALI_RE = re.compile(
    r'['
    r'\u0900-\u097F'   # Devanagari (Hindi etc.)
    r'\u0600-\u06FF'   # Arabic
    r'\u0A00-\u0A7F'   # Gurmukhi
    r'\u0A80-\u0AFF'   # Gujarati
    r'\u0B00-\u0B7F'   # Odia
    r'\u0B80-\u0BFF'   # Tamil
    r'\u0C00-\u0C7F'   # Telugu
    r'\u0C80-\u0CFF'   # Kannada
    r'\u0D00-\u0D7F'   # Malayalam
    r'\u3000-\u9FFF'   # CJK
    r'\uAC00-\uD7AF'   # Korean
    r']'
)

def remove_non_bengali_scripts(text: str) -> str:
    """
    Remove characters from non-Bengali scripts (Devanagari, Arabic, CJK, etc.)
    that appear as faster-whisper decoding artifacts.
    Preserves: Bengali (U+0980–U+09FF), ASCII, digits, spaces, punctuation.
    """
    text = _NON_BENGALI_RE.sub('', text)
    # Clean up any resulting double spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# ─── Master Post-processing Function ────────────────────────────────────────

def postprocess(text: str, cfg: PostProcessConfig = None) -> str:
    """Apply full Bengali post-processing pipeline."""
    if cfg is None:
        cfg = PostProcessConfig()

    if not text or not text.strip():
        return ""

    # Stage 1: Unicode normalization
    text = normalize_unicode(text, cfg.unicode_norm)

    # Stage 2: Bengali-specific normalization
    if cfg.use_bn_unicode_normalizer:
        text = normalize_bengali_unicode(text)

    # Stage 3: Zero-width character cleanup
    if cfg.clean_zero_width:
        text = clean_zero_width(text)

    # Stage 4: Hasanta normalization
    text = normalize_hasanta(text)

    # Stage 5: Hallucination blacklist
    if cfg.hallucination_filter:
        text = filter_hallucination_phrases(text, cfg.hallucination_blacklist)

    # Stage 6: De-looping
    if cfg.deloop:
        text = deloop_text(
            text,
            min_ngram=cfg.deloop_min_ngram,
            max_ngram=cfg.deloop_max_ngram,
            max_repeat=cfg.deloop_max_repeat,
        )

    # Stage 7: Consecutive duplicate removal
    if cfg.remove_consecutive_duplicates:
        text = remove_consecutive_duplicates(text)

    # Stage 8: Bengali number-to-word (if enabled)
    if cfg.normalize_bengali_numbers:
        text = bengali_number_to_words(text)

    # Stage 9: Bengali character cleanup
    text = clean_bengali_text(
        text,
        remove_english=cfg.remove_english,
        strip_punctuation=cfg.strip_punctuation,
    )

    # Stage 10: Orphan diacritics cleanup
    text = clean_orphan_diacritics(text)

    # Stage 11: Non-Bengali script removal (Devanagari, Arabic, etc.)
    text = remove_non_bengali_scripts(text)

    # Final whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# ## 9. Single-file Transcription Orchestrator 
# (runs the whole end-to-end pipeline for one audio file)
# 


def transcribe_file(
    audio_path: str,
    cfg: PipelineConfig = None,
    verbose: bool = True,
) -> str:
    """
    Full pipeline for a single audio file.
    Demucs → Load → (Denoise) → VAD → Segment → Transcribe → PostProcess
    Returns cleaned Bengali transcript.
    """
    if cfg is None:
        cfg = config  # use global config

    total_t0 = time.time()
    audio_path = str(audio_path)

    # ── Pre-warm faster-whisper models on all GPUs while CPU does I/O ──
    if cfg.whisper.model_mode == "faster":
        _get_faster_whisper_models(cfg.whisper)

    if verbose:
        print("=" * 60)
        print(f"  Bengali ASR Pipeline (HF Whisper) [{cfg.whisper.model_mode.upper()} mode]")
        print(f"  File: {Path(audio_path).name}")
        print("=" * 60)

    # ── Step 0: Demucs vocal separation ──
    actual_audio_path = audio_path
    if cfg.audio.use_demucs:
        try:
            actual_audio_path = separate_vocals(
                audio_path,
                device=cfg.whisper.device,
            )
        except Exception as e:
            print(f"[WARN] Demucs failed: {e}, using original audio")

    # ── Step 1: Load audio ──
    if verbose:
        print(f"\n[1/5] Loading audio...")
    audio, sr = load_audio(actual_audio_path, cfg.audio.sample_rate)
    duration_min = len(audio) / sr / 60
    if verbose:
        print(f"       Duration: {duration_min:.1f} min | Samples: {len(audio):,}")

    # ── Step 2: Optional denoising ──
    if cfg.audio.denoise:
        if verbose:
            print(f"\n[2/5] Denoising...")
        audio = denoise_audio(
            audio, sr,
            stationary=cfg.audio.denoise_stationary,
            prop_decrease=cfg.audio.denoise_prop_decrease,
            n_fft=cfg.audio.denoise_n_fft,
            hop_length=cfg.audio.denoise_hop_length,
        )
    elif verbose:
        print(f"\n[2/5] Denoising: SKIPPED")

    # ── Step 3: VAD ──
    if verbose:
        print(f"\n[3/5] Running VAD...")
    vad_timestamps = run_vad(
        audio, sr,
        threshold=cfg.vad.threshold,
        min_speech_duration_ms=cfg.vad.min_speech_duration_ms,
        min_silence_duration_ms=cfg.vad.min_silence_duration_ms,
        max_speech_duration_s=cfg.vad.max_speech_duration_s,
        speech_pad_ms=cfg.vad.speech_pad_ms,
        window_size_samples=cfg.vad.window_size_samples,
    )

    if not vad_timestamps:
        if verbose:
            print("[WARN] No speech segments found!")
        return ""

    # ── Step 4: Segmentation / Stitching ──
    if cfg.stitched.enabled:
        # ── Stitched mode: join VAD regions with short silence, let pipeline chunk ──
        if verbose:
            print(f"\n[4/5] Stitching VAD regions (mode=stitched)...")
        stitched_audio = stitch_vad_audio(audio, vad_timestamps, sr, cfg.stitched)

        if len(stitched_audio) == 0:
            if verbose:
                print("[WARN] Stitched audio is empty!")
            return ""

        # Save stitched audio with source file identity
        audio_stem = Path(audio_path).stem
        output_dir = Path("output")
        output_dir.mkdir(exist_ok=True)
        stitched_filename = f"stitched_{audio_stem}.wav"
        stitched_save_path = output_dir / stitched_filename
        
        # Convert to tensor for torchaudio
        if isinstance(stitched_audio, np.ndarray):
            stitched_tensor = torch.from_numpy(stitched_audio)
        else:
            stitched_tensor = stitched_audio
            
        # Ensure (C, T) shape
        if stitched_tensor.dim() == 1:
            stitched_tensor = stitched_tensor.unsqueeze(0)
        
        # Ensure float32 for torchaudio save
        if stitched_tensor.dtype != torch.float32:
            stitched_tensor = stitched_tensor.float()
            
        torchaudio.save(str(stitched_save_path), stitched_tensor, sr)
        if verbose:
            print(f"  Saved stitched audio → {stitched_save_path}")
       
        
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        
        # ── Step 5: Transcription (load saved WAV via librosa for consistency) ──
        if verbose:
            print(f"\n[5/5] Transcribing stitched audio (pipeline-chunked)...")
        if cfg.whisper.model_mode == "faster":
            raw_text = transcribe_stitched_faster(
                str(stitched_save_path), cfg.whisper, cfg.stitched, verbose=verbose
            )
        else:
            raw_text = transcribe_stitched(
                str(stitched_save_path), cfg.whisper, cfg.stitched, verbose=verbose
            )
    else:
        # ── Original mode: segment-by-segment ──
        if verbose:
            print(f"\n[4/5] Creating segments...")
        segments = create_segments(
            audio, vad_timestamps, sr,
            target_duration_s=cfg.segmentation.target_duration_s,
            max_duration_s=cfg.segmentation.max_duration_s,
            min_duration_s=cfg.segmentation.min_duration_s,
            overlap_s=cfg.segmentation.overlap_s,
            merge_gap_s=cfg.segmentation.merge_gap_s,
        )

        if not segments:
            if verbose:
                print("[WARN] No valid segments created!")
            return ""

        # ── Step 5: Transcription ──
        if verbose:
            print(f"\n[5/5] Transcribing {len(segments)} segments...")

        if cfg.whisper.model_mode == "faster":
            raw_text, seg_results = transcribe_segments_faster(segments, cfg.whisper, verbose=verbose)
        else:
            raw_text, seg_results = transcribe_segments(segments, cfg.whisper, verbose=verbose)

    # ── Cleanup: Remove Demucs cache for this file ──
    if cfg.audio.use_demucs:
        _cleanup_demucs_cache(audio_path)

    # ── Post-processing ──
    if verbose:
        print(f"\n[Post] Cleaning text...")
    final_text = postprocess(raw_text, cfg.postprocess)

    total_elapsed = time.time() - total_t0
    audio_duration = len(audio) / sr

    if verbose:
        rtf = total_elapsed / audio_duration if audio_duration > 0 else 0
        print(f"\n{'=' * 60}")
        print(f"  ✓ Done in {total_elapsed:.1f}s (RTF: {rtf:.2f}x)")
        print(f"  Audio: {audio_duration:.1f}s | Output: {len(final_text)} chars")
        print(f"{'=' * 60}")

    return final_text

# ## 10. Evaluation on the Training set (computing WER) 


def evaluate_on_train(
    train_audio_dir: str = None,
    train_annotation_dir: str = None,
    output_csv: str = "evaluation_results.csv",
    cfg: PipelineConfig = None,
    num_samples: int = -1,
    verbose: bool = True,
) -> Dict:
    """
    Evaluate pipeline on train set with ground truth annotations.
    
    Args:
        train_audio_dir: Path to train/audio/ (defaults to config)
        train_annotation_dir: Path to train/annotation/ with *.txt files (defaults to config).
                              If None or non-existent, only transcribes without WER.
        output_csv: Path to save per-file evaluation results (CSV).
        num_samples: -1 for all, or limit for quick eval
    
    Returns dict with per-file and aggregate WER metrics.
    """
    if cfg is None:
        cfg = config

    if train_audio_dir is None:
        train_audio_dir = cfg.train_audio_dir
    if train_annotation_dir is None:
        train_annotation_dir = cfg.train_annotation_dir

    audio_dir = Path(train_audio_dir)
    annotation_dir = Path(train_annotation_dir) if train_annotation_dir else None

    # Find audio files
    audio_files = sorted(audio_dir.glob("*.wav"))
    if not audio_files:
        audio_files = sorted(audio_dir.glob("*.mp3")) + sorted(audio_dir.glob("*.flac"))

    if not audio_files:
        print(f"[ERROR] No audio files found in {audio_dir}")
        return {}

    # Find matching pairs (if annotations exist)
    has_annotations = annotation_dir and annotation_dir.is_dir()
    pairs = []
    for audio_path in audio_files:
        stem = audio_path.stem
        gt_path = annotation_dir / f"{stem}.txt" if has_annotations else None
        has_gt = gt_path and gt_path.exists()
        pairs.append((audio_path, gt_path if has_gt else None))

    if num_samples > 0:
        pairs = pairs[:num_samples]

    # Try to import jiwer
    try:
        from jiwer import wer as compute_wer
        has_jiwer = True
    except ImportError:
        print("[WARN] jiwer not installed — using basic WER")
        has_jiwer = False

    gt_found = sum(1 for _, gt in pairs if gt is not None)
    print(f"\n{'#' * 60}")
    print(f"  Train Evaluation: {len(pairs)} files ({gt_found} with GT)")
    print(f"  Output CSV: {output_csv}")
    print(f"{'#' * 60}\n")

    # Initialize CSV
    output_path = Path(output_csv)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["file_name", "wer", "transcription"])

    results = {}
    all_hypotheses = []
    all_references = []
    eval_t0 = time.time()

    for i, (audio_path, gt_path) in enumerate(pairs):
        stem = audio_path.stem
        print(f"\n{'─' * 40} [{i + 1}/{len(pairs)}] {stem}")

        # Load ground truth
        gt_text = ""
        if gt_path:
            gt_text = gt_path.read_text(encoding="utf-8").strip()

        # Transcribe
        try:
            hypothesis = transcribe_file(str(audio_path), cfg=cfg, verbose=verbose)
        except Exception as e:
            print(f"[ERROR] Failed on {stem}: {e}")
            hypothesis = ""
                
        # WER
        file_wer = None
        if gt_text and hypothesis:
            if has_jiwer:
                file_wer = compute_wer(gt_text, hypothesis)
            else:
                file_wer = _basic_wer(gt_text, hypothesis)
            all_references.append(gt_text)
            all_hypotheses.append(hypothesis)

        results[stem] = {
            "wer": file_wer,
            "gt_chars": len(gt_text) if gt_text else 0,
            "hyp_chars": len(hypothesis),
            "hypothesis": hypothesis[:200],
        }

        # Save to CSV
        try:
            with open(output_path, "a", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)
                writer.writerow([stem, file_wer if file_wer is not None else "", hypothesis])
        except Exception as e:
            print(f"[WARN] Failed to write to CSV for {stem}: {e}")

        if file_wer is not None:
            print(f"  WER: {file_wer:.4f} ({file_wer * 100:.2f}%)")
        print(f"  GT: {len(gt_text)} chars | Hyp: {len(hypothesis)} chars")

        # Memory cleanup
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Overall WER
    overall_wer = None
    if all_references and all_hypotheses:
        if has_jiwer:
            overall_wer = compute_wer(all_references, all_hypotheses)
        else:
            wers = [r["wer"] for r in results.values() if r["wer"] is not None]
            overall_wer = np.mean(wers) if wers else None

    eval_elapsed = time.time() - eval_t0

    print(f"\n{'=' * 60}")
    print(f"  EVALUATION RESULTS")
    print(f"{'=' * 60}")
    print(f"  Files evaluated: {len(results)}")
    if overall_wer is not None:
        print(f"  Overall WER: {overall_wer:.4f} ({overall_wer * 100:.2f}%)")
        wers = [r["wer"] for r in results.values() if r["wer"] is not None]
        if wers:
            print(f"  Min WER: {min(wers):.4f} | Max: {max(wers):.4f} | Median: {np.median(wers):.4f}")
    else:
        print(f"  WER: N/A (no ground truth annotations found)")
    print(f"  Time: {eval_elapsed:.1f}s")

    print(f"\n  Per-file breakdown:")
    for stem in sorted(results.keys()):
        r = results[stem]
        wer_str = f"WER={r['wer']:.4f}" if r["wer"] is not None else "WER=N/A"
        print(f"    {stem}: {wer_str} | hyp={r['hyp_chars']} chars")

    print(f"{'=' * 60}\n")

    return {
        "overall_wer": overall_wer,
        "per_file": results,
        "num_files": len(results),
        "eval_time_s": eval_elapsed,
    }



def _basic_wer(reference: str, hypothesis: str) -> float:
    """Simple WER (Levenshtein at word level) when jiwer is unavailable."""
    ref_words = reference.split()
    hyp_words = hypothesis.split()
    if not ref_words:
        return 1.0 if hyp_words else 0.0
    n, m = len(ref_words), len(hyp_words)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if ref_words[i - 1] == hyp_words[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])
    return dp[n][m] / n


# ## 11. Generating the final submission CSV from Test Set


def generate_submission(
    test_audio_dir: str = None,
    output_csv: str = "submission.csv",
    cfg: PipelineConfig = None,
    file_range: Optional[str] = None,
    verbose: bool = True,
) -> Dict[str, str]:
    """
    Generate submission CSV by transcribing all files in test directory.
    
    Crash-resilient: appends each result to CSV immediately.
    Supports resume from existing partial CSV.
    
    Args:
        test_audio_dir: Path to test/audio/ directory (defaults to config)
        output_csv: Output CSV path
        file_range: Optional "start-end" string (1-indexed inclusive), e.g. "1-10"
        verbose: Print progress
    
    Returns:
        Dict mapping filename → transcript
    """
    if cfg is None:
        cfg = config
    if test_audio_dir is None:
        test_audio_dir = cfg.test_audio_dir

    audio_dir = Path(test_audio_dir)
    audio_files = sorted(audio_dir.glob("*.wav"))
    if not audio_files:
        audio_files = sorted(audio_dir.glob("*.mp3")) + sorted(audio_dir.glob("*.flac"))

    if not audio_files:
        print(f"[ERROR] No audio files found in {audio_dir}")
        return {}

    # Apply range filter
    total_found = len(audio_files)
    if file_range:
        parts = str(file_range).split("-")
        if len(parts) == 2:
            start_idx = max(0, int(parts[0]) - 1)
            end_idx = min(len(audio_files), int(parts[1]))
            audio_files = audio_files[start_idx:end_idx]
            print(f"[Range] Processing files {start_idx + 1}-{end_idx} of {total_found}")

    # Load existing CSV for resume
    output_path = Path(output_csv)
    existing_results = {}
    if output_path.exists():
        with open(output_path, "r", encoding="utf-8") as f:
            reader = csv.reader(f)
            header = next(reader, None)
            if header:
                for row in reader:
                    if len(row) >= 2:
                        existing_results[row[0].strip()] = row[1]
        print(f"[Resume] Found {len(existing_results)} existing results in {output_csv}")

    # Ensure CSV with header exists
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if not output_path.exists() or not existing_results:
        with open(output_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["filename", "transcript"])

    remaining = len(audio_files) - len(
        set(af.stem for af in audio_files) & set(existing_results.keys())
    )

    print(f"\n{'#' * 60}")
    print(f"  Submission Generation: {len(audio_files)} files")
    print(f"  Already done: {len(existing_results)} | Remaining: {remaining}")
    print(f"  Output: {output_csv}")
    print(f"  Model mode: {cfg.whisper.model_mode.upper()}")
    if cfg.whisper.model_mode == "merged":
        print(f"  Model: {cfg.whisper.merged_model_path}")
    elif cfg.whisper.model_mode == "adapter":
        print(f"  Base: {cfg.whisper.model_id}")
        print(f"  Adapter: {cfg.whisper.adapter_path}")
    else:
        print(f"  Model: {cfg.whisper.model_id}")
    print(f"{'#' * 60}\n")

    results = dict(existing_results)
    batch_t0 = time.time()
    processed = 0
    skipped = 0

    for i, audio_path in enumerate(audio_files):
        filename = audio_path.stem

        # Skip if already done
        if filename in existing_results:
            skipped += 1
            if verbose:
                print(f"[{i + 1}/{len(audio_files)}] SKIP {filename} (already in CSV)")
            continue

        print(f"\n{'─' * 40} [{i + 1}/{len(audio_files)}] {audio_path.name}")

        try:
            text = transcribe_file(str(audio_path), cfg=cfg, verbose=verbose)
            results[filename] = text
            processed += 1

            # Append immediately (crash-resilient)
            with open(output_path, "a", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)
                writer.writerow([filename, text])

            if verbose:
                print(f"  ✓ Appended {filename} to {output_csv} ({len(text)} chars)")

        except Exception as e:
            print(f"[ERROR] Failed on {audio_path.name}: {e}")
            results[filename] = ""
            with open(output_path, "a", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)
                writer.writerow([filename, ""])

        # Memory + disk cleanup
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        _enforce_stitched_cache_limit()  # keep stitched audio cache ≤ 5 GB

    batch_elapsed = time.time() - batch_t0
    print(f"\n{'#' * 60}")
    print(f"  ✓ Submission complete: {processed} transcribed, {skipped} skipped")
    print(f"  Total in CSV: {len(results)} | Time: {batch_elapsed:.1f}s")
    print(f"  Saved to: {output_csv}")
    print(f"{'#' * 60}\n")

    return results

# # ═══════════════════════════════════════════════════════════════════════════
# # MODEL MODE SELECTION — Uncomment ONE of the three blocks below
# # ═══════════════════════════════════════════════════════════════════════════
#
# ── MODE 1: Base model (original, no fine-tuning) ──
# config.whisper.model_mode = "base"
# config.whisper.model_id = "/kaggle/input/datasets/faysal314mahmud/tugstugi-bengaliai-asr-whisper-medium/tugstugi_bengaliai-asr_whisper-medium"
#
# ── MODE 2: Merged model (LoRA baked into base — recommended for speed) ──
# config.whisper.model_mode = "merged"
# config.whisper.merged_model_path = "/kaggle/input/datasets/faysal314/whisper-md-lora-e5-bcfdata/runpod_download/merged_model"
#
# ── MODE 3: Adapter mode (base + LoRA adapter — recommended for smaller upload) ──
# config.whisper.model_mode = "adapter"
# config.whisper.model_id = "bengaliAI/tugstugi_bengaliai-asr_whisper-medium"
# config.whisper.adapter_path = "/kaggle/input/whisper-md-lora/runpod_download/final_adapter"
#
# ── MODE 4: Faster Whisper (CTranslate2 — fastest inference) ──
config.whisper.model_mode = "faster"
config.whisper.faster_model_path = "/kaggle/input/datasets/faysal314/whisper-md-lora-ep7-ct2/whisper_md_lora_e7_ct2"
config.whisper.faster_num_gpus = 2       # Use both T4s on Kaggle
config.whisper.faster_batch_size = 32   # Fill GPU memory with batched inference


# ── Set active mode here ──
# config.whisper.model_mode = "base"  # ← change to "merged" or "adapter"

# print(f"\n[Active Mode] {config.whisper.model_mode.upper()}")


# # ── Evaluate on train set (WER report) ──

# eval_results = evaluate_on_train(
#     train_audio_dir=config.train_audio_dir,
#     train_annotation_dir=config.train_annotation_dir,
#     output_csv="eval_train.csv",
#     cfg=config,
#     num_samples=1,  # -1 for all, or e.g. 3 for quick test
#     verbose=True,
# )

# # ── Generate submission CSV from test set ──

results = generate_submission(
    test_audio_dir=config.test_audio_dir,
    output_csv="submission_LoRA_Merged_nodenoise_e7_ct2_t4.csv",
    cfg=config,
    file_range=None,    # e.g., "1-10" for partial run
    verbose=True,
)

[Config] Base data dir: /kaggle/input/competitions/dl-sprint-4-0-bengali-long-form-speech-recognition/transcription/transcription
[Config] Train audio: /kaggle/input/competitions/dl-sprint-4-0-bengali-long-form-speech-recognition/transcription/transcription/train/audio
[Config] Test audio: /kaggle/input/competitions/dl-sprint-4-0-bengali-long-form-speech-recognition/transcription/transcription/test/audio
[Config] Model mode: base
[Config] Model: 
[Config] Device: cuda

############################################################
  Submission Generation: 24 files
  Already done: 0 | Remaining: 24
  Output: submission_LoRA_Merged_nodenoise_e7_ct2_t4.csv
  Model mode: FASTER
  Model: 
############################################################


──────────────────────────────────────── [1/24] test_001.wav
[Whisper-Faster] Loading CTranslate2 model on 2 GPU(s)...
  GPU 0: loaded in 18.7s
  GPU 1: loaded in 4.8s
[Whisper-Faster] All 2 GPU(s) ready in 23.5s | compute_type=float16
  Bengali 